# VietNamNet News — Crawling Data

Notebook crawl dữ liệu bài viết từ VietNamNet cho **19 chuyên mục**.

| Bước | Nội dung |
|------|----------|
| **Section 1** | Chuẩn bị: kiểm tra thư viện, cấu hình, hàm tiện ích |
| **Section 2** | Crawl URL → lưu `Dataset/data_URLs.json` |
| **Section 3** | Crawl title & content → lưu `Dataset/<category>.parquet` |

> **Logic thông minh (per-category):**  
> - Section 2 chỉ crawl URL cho những category **chưa có** trong `data_URLs.json`.  
> - Section 3 chỉ crawl content cho những category **chưa có** file `.parquet`.  
>   Nếu category đó chưa có URL, sẽ tự crawl URL trước rồi mới crawl content.

---
## Section 1: Chuẩn bị

In [1]:
import importlib

# ── Kiem tra tat ca thu vien can thiet ───────────────────────────────────────
REQUIRED = {
    "requests":     "requests",
    "tqdm":         "tqdm",
    "bs4":          "beautifulsoup4",
    "lxml":         "lxml",
    "pyarrow":      "pyarrow",
    "aiohttp":      "aiohttp",
    "newspaper":    "newspaper3k",
    "trafilatura":  "trafilatura",
    "readability":  "readability-lxml",
    "goose3":       "goose3",
}

print("Thu vien            | Trang thai")
print("-" * 36)
missing = []
for mod, pkg in REQUIRED.items():
    ok = importlib.util.find_spec(mod) is not None
    mark = "✅" if ok else "❌  CHUA CAI"
    print(f"  {pkg:<20} {mark}")
    if not ok:
        missing.append(pkg)

if missing:
    print()
    print("❌ Thieu thu vien. Chay lenh sau roi khoi dong lai kernel:")
    print(f"\n    pip install {chr(32).join(missing)}\n")
    raise ImportError(f"Thieu: {chr(44).join(missing)}")

# ── Import ────────────────────────────────────────────────────────────────────
import os
import json
import asyncio
import aiohttp
import tqdm
import requests
import pyarrow as pa
import pyarrow.parquet as pq
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import datetime

print()
print("✅ Tat ca thu vien da san sang.")


Thu vien            | Trang thai
------------------------------------
  requests             ✅
  tqdm                 ✅
  beautifulsoup4       ✅
  lxml                 ✅
  pyarrow              ✅
  aiohttp              ✅
  newspaper3k          ✅
  trafilatura          ✅
  readability-lxml     ✅
  goose3               ✅

✅ Tat ca thu vien da san sang.


d:\python\python_3.14\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
# Cấu hình
MAX_PAGES   = 500   # Số trang tối đa crawl URL mỗi category
                    # Ví dụ: MAX_PAGES=500 → crawl từ ban-doc đến ban-doc-page499
BATCH_SIZE  = 100   # Số bài ghi vào parquet mỗi lần
NUM_WORKERS = 32   # Số luồng crawl song song

# Đường dẫn
DATASET_DIR    = os.path.normpath(os.path.join(os.getcwd(), "..", "Dataset"))
DATA_URLS_PATH = os.path.join(DATASET_DIR, "data_URLs.json")

# 19 chuyên mục cần crawl
CATEGORIES = [
    "chinh-tri",
    "thoi-su",
    "kinh-doanh",
    "dan-toc-ton-giao",
    "the-thao",
    "giao-duc",
    "the-gioi",
    "doi-song",
    "van-hoa-giai-tri",
    "suc-khoe",
    "cong-nghe",
    "phap-luat",
    "oto-xe-may",
    "du-lich",
    "bat-dong-san",
    "ban-doc",
    "tuan-viet-nam",
    "bao-ve-nguoi-tieu-dung",
    "thi-truong-tieu-dung",
]

print(f"Dataset dir : {DATASET_DIR}")
print(f"URLs file   : {DATA_URLS_PATH}")
print(f"Categories  : {len(CATEGORIES)}")

Dataset dir : d:\VietNamNet-News-Classification\Dataset
URLs file   : d:\VietNamNet-News-Classification\Dataset\data_URLs.json
Categories  : 19


In [3]:
# Khởi tạo thư mục Dataset và file data_URLs.json nếu chưa tồn tại
os.makedirs(DATASET_DIR, exist_ok=True)

if not os.path.exists(DATA_URLS_PATH):
    with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
        json.dump({}, f)
    print(f"Đã tạo: {DATA_URLS_PATH}")
else:
    # Nếu file tồn tại nhưng rỗng hoặc lỗi, reset về {}
    try:
        with open(DATA_URLS_PATH, encoding="utf-8") as f:
            content = f.read().strip()
        if not content:
            raise ValueError("File rỗng")
        json.loads(content)  # validate JSON
    except Exception:
        with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
            json.dump({}, f)
        print(f"data_URLs.json bị lỗi/rỗng → đã reset về {{}}: {DATA_URLS_PATH}")

print(f"✅ Dataset dir : {DATASET_DIR}")
print(f"✅ URLs file   : {DATA_URLS_PATH}")

✅ Dataset dir : d:\VietNamNet-News-Classification\Dataset
✅ URLs file   : d:\VietNamNet-News-Classification\Dataset\data_URLs.json


In [4]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


def get_urls_of_category(category, max_pages=MAX_PAGES, silent=False):
    session = requests.Session()
    urls = []
    _iter = range(1, max_pages + 1)
    if not silent:
        _iter = tqdm.tqdm(_iter, desc=category, leave=False)
    for page in _iter:
        page_url = (
            f"https://vietnamnet.vn/{category}"
            if page == 1
            else f"https://vietnamnet.vn/{category}-page{page - 1}"
        )
        try:
            content = session.get(page_url, headers=HEADERS, timeout=10).content
        except Exception:
            break
        soup = BeautifulSoup(content, "lxml")
        titles = soup.find_all(class_="vnn-title")
        if len(titles) == 0:
            break
        for title in titles:
            a = title if title.name == "a" else title.find("a")
            if a:
                href = a.get("href", "")
                if href.startswith("/"):
                    href = "https://vietnamnet.vn" + href
                if href:
                    urls.append(href)
    return urls


def _fetch_html(url):
    """Tai HTML voi requests + HEADERS, tra ve (bytes, encoding) hoac (None, None)."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        resp.encoding = resp.apparent_encoding or "utf-8"
        return resp.content, resp.encoding
    except Exception:
        return None, None


def extract_content(url):
    """Trich xuat (title, content) tu URL bai viet VietNamNet.

    Thu lan luot 5 phuong phap theo do uu tien:
      1. BeautifulSoup  - selector chinh xac nhat cho VietNamNet
      2. trafilatura    - article extractor chinh xac, ho tro tieng Viet
      3. readability    - Firefox Reader Mode (readability-lxml)
      4. goose3         - structured article extractor
      5. newspaper3k    - fallback cuoi cung

    Tra ve (title, content) luon la str, khong tra None.
    """
    title, content = "", ""

    # -- Lay HTML mot lan, dung lai cho BS4 + trafilatura + readability ------
    html_bytes, encoding = _fetch_html(url)

    # -- 1. BeautifulSoup (VietNamNet-specific selectors) --------------------
    if html_bytes is not None:
        try:
            soup = BeautifulSoup(html_bytes, "lxml")
            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()

            for sel in [
                "h1.content-detail-title", "h1.ArticleTitle",
                "h1.title-detail", "h1.title", "h1",
            ]:
                el = soup.select_one(sel)
                if el:
                    t = el.get_text(strip=True)
                    if len(t) > 5:
                        title = t
                        break

            for sel in [
                "div.ArticleContent", "div.content-detail-body",
                "div.maincontent", "div.main-content",
                "article", "main",
            ]:
                el = soup.select_one(sel)
                if el:
                    paras = [
                        p.get_text(strip=True) for p in el.find_all("p")
                        if len(p.get_text(strip=True)) > 20
                    ]
                    if paras:
                        content = " ".join(paras)
                        break

            if not content:
                content = " ".join(
                    p.get_text(strip=True) for p in soup.find_all("p")
                    if len(p.get_text(strip=True)) > 30
                )
        except Exception:
            pass

    # -- 2. trafilatura -------------------------------------------------------
    if (not title or not content) and html_bytes is not None:
        try:
            import trafilatura
            html_str = html_bytes.decode(encoding or "utf-8", errors="replace")
            result = trafilatura.extract(
                html_str,
                include_comments=False,
                include_tables=False,
                favor_precision=True,
                output_format="txt",
            )
            meta = trafilatura.extract_metadata(html_str)
            if not title and meta and meta.title:
                title = meta.title.strip()
            if not content and result:
                content = result.strip()
        except Exception:
            pass

    # -- 3. readability-lxml --------------------------------------------------
    if (not title or not content) and html_bytes is not None:
        try:
            from readability import Document
            doc = Document(html_bytes.decode(encoding or "utf-8", errors="replace"))
            if not title:
                t = doc.title()
                if t and len(t) > 5:
                    title = t.strip()
            if not content:
                summary_html = doc.summary(html_partial=True)
                summary_soup = BeautifulSoup(summary_html, "lxml")
                paras = [
                    p.get_text(strip=True) for p in summary_soup.find_all("p")
                    if len(p.get_text(strip=True)) > 20
                ]
                if paras:
                    content = " ".join(paras)
        except Exception:
            pass

    # -- 4. goose3 ------------------------------------------------------------
    if not title or not content:
        try:
            from goose3 import Goose
            with Goose({"browser_user_agent": HEADERS["User-Agent"], "request_timeout": 12}) as g:
                article = g.extract(url=url)
                if not title and article.title:
                    title = article.title.strip()
                if not content and article.cleaned_text:
                    content = article.cleaned_text.strip()
        except Exception:
            pass

    # -- 5. newspaper3k -------------------------------------------------------
    if not title or not content:
        try:
            from newspaper import Article
            article = Article(url, language="vi")
            article.download()
            article.parse()
            if not title:
                title = article.title or ""
            if not content:
                content = article.text or ""
        except Exception:
            pass

    return title.strip(), content.strip()


SCHEMA = pa.schema([
    pa.field("class",   pa.string()),
    pa.field("url",     pa.string()),
    pa.field("title",   pa.string()),
    pa.field("content", pa.string()),
])


def flush_batch(writer, batch):
    if not batch:
        return
    table = pa.table(
        {
            "class":   [r["class"]   for r in batch],
            "url":     [r["url"]     for r in batch],
            "title":   [r["title"]   for r in batch],
            "content": [r["content"] for r in batch],
        },
        schema=SCHEMA,
    )
    writer.write_table(table)
    batch.clear()


def fmt_dur(seconds):
    s = int(seconds)
    h, rem = divmod(s, 3600)
    m, s   = divmod(rem, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    if m > 0: return f"{m}m {s}s"
    return f"{s}s"


def now():
    return datetime.datetime.now().strftime("%H:%M:%S")




# ── Async helpers (aiohttp) ──────────────────────────────────────────────────

async def _fetch_html_async(session, url):
    """Tai HTML bang aiohttp, tra ve (bytes, encoding) hoac (None, None)."""
    try:
        async with session.get(url) as resp:
            html_bytes = await resp.read()
            encoding = resp.charset or "utf-8"
            return html_bytes, encoding
    except Exception:
        return None, None


async def extract_content_async(session, url):
    """Phien ban async cua extract_content.
    Tai HTML mot lan bang aiohttp, chia se cho tat ca 5 layer.
    Layer CPU-bound (goose3, newspaper3k) chay trong thread executor.
    """
    title, content = "", ""
    loop = asyncio.get_event_loop()

    # Fetch HTML mot lan duy nhat
    html_bytes, encoding = await _fetch_html_async(session, url)
    html_str = html_bytes.decode(encoding or "utf-8", errors="replace") if html_bytes else None

    # -- 1. BeautifulSoup -----------------------------------------------------
    if html_bytes is not None:
        try:
            soup = BeautifulSoup(html_bytes, "lxml")
            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()
            for sel in [
                "h1.content-detail-title", "h1.ArticleTitle",
                "h1.title-detail", "h1.title", "h1",
            ]:
                el = soup.select_one(sel)
                if el:
                    t = el.get_text(strip=True)
                    if len(t) > 5:
                        title = t
                        break
            for sel in [
                "div.ArticleContent", "div.content-detail-body",
                "div.maincontent", "div.main-content",
                "article", "main",
            ]:
                el = soup.select_one(sel)
                if el:
                    paras = [
                        p.get_text(strip=True) for p in el.find_all("p")
                        if len(p.get_text(strip=True)) > 20
                    ]
                    if paras:
                        content = " ".join(paras)
                        break
            if not content:
                content = " ".join(
                    p.get_text(strip=True) for p in soup.find_all("p")
                    if len(p.get_text(strip=True)) > 30
                )
        except Exception:
            pass

    # -- 2. trafilatura -------------------------------------------------------
    if (not title or not content) and html_str is not None:
        try:
            import trafilatura
            def _traf():
                r = trafilatura.extract(
                    html_str, include_comments=False, include_tables=False,
                    favor_precision=True, output_format="txt",
                )
                m = trafilatura.extract_metadata(html_str)
                return (m.title.strip() if m and m.title else ""), (r.strip() if r else "")
            tt, tc = await loop.run_in_executor(None, _traf)
            if not title: title = tt
            if not content: content = tc
        except Exception:
            pass

    # -- 3. readability-lxml --------------------------------------------------
    if (not title or not content) and html_str is not None:
        try:
            def _read():
                from readability import Document
                doc = Document(html_str)
                t = (doc.title() or "").strip()
                s_soup = BeautifulSoup(doc.summary(html_partial=True), "lxml")
                paras = [
                    p.get_text(strip=True) for p in s_soup.find_all("p")
                    if len(p.get_text(strip=True)) > 20
                ]
                return t, " ".join(paras)
            rt, rc = await loop.run_in_executor(None, _read)
            if not title and len(rt) > 5: title = rt
            if not content: content = rc
        except Exception:
            pass

    # -- 4. goose3 (dung lai html_str, khong fetch lai) -----------------------
    if not title or not content:
        try:
            def _goose():
                from goose3 import Goose
                with Goose({"browser_user_agent": HEADERS["User-Agent"], "request_timeout": 12}) as g:
                    a = g.extract(raw_html=html_str, url=url) if html_str else g.extract(url=url)
                    return (a.title or "").strip(), (a.cleaned_text or "").strip()
            gt, gc = await loop.run_in_executor(None, _goose)
            if not title: title = gt
            if not content: content = gc
        except Exception:
            pass

    # -- 5. newspaper3k (dung lai html_str, khong fetch lai) ------------------
    if not title or not content:
        try:
            def _news():
                from newspaper import Article
                a = Article(url, language="vi")
                if html_str:
                    a.set_html(html_str)
                else:
                    a.download()
                a.parse()
                return (a.title or "").strip(), (a.text or "").strip()
            nt, nc = await loop.run_in_executor(None, _news)
            if not title: title = nt
            if not content: content = nc
        except Exception:
            pass

    return title.strip(), content.strip()
print("Da dinh nghia xong cac ham tien ich.")


Da dinh nghia xong cac ham tien ich.


---
## Section 2: Crawl URL

Crawl danh sách URL bài viết từ VietNamNet và lưu vào `Dataset/data_URLs.json`.

**Chỉ crawl những category chưa có URL trong file.** Category đã có URL sẽ được giữ nguyên.

In [5]:
# Load dữ liệu URL đã có (nếu có)
if os.path.exists(DATA_URLS_PATH):
    try:
        with open(DATA_URLS_PATH, encoding="utf-8") as f:
            all_urls = json.load(f)
    except Exception:
        all_urls = {}
else:
    all_urls = {}

# Xác định category nào cần crawl URL
missing_url_cats = [
    cat for cat in CATEGORIES
    if cat not in all_urls or len(all_urls[cat]) == 0
]

if not missing_url_cats:
    print("✅ Tất cả categories đã có URL. Bỏ qua bước crawl URL.\n")
    for cat in CATEGORIES:
        print(f"  {cat}: {len(all_urls.get(cat, []))} URLs")
else:
    t0_total = time.time()
    print(f"[{now()}] Cần crawl URL cho {len(missing_url_cats)} category:\n")
    for cat in missing_url_cats:
        print(f"  • {cat}")
    print()

    for i, category in enumerate(missing_url_cats, 1):
        t0 = time.time()
        print(f"[{now()}] [{i}/{len(missing_url_cats)}] Bắt đầu crawl URL: {category}")
        urls = get_urls_of_category(category, max_pages=MAX_PAGES)
        all_urls[category] = urls
        dur = time.time() - t0
        print(f"[{now()}] ✓ {category}: {len(urls)} URLs  ({fmt_dur(dur)})")

    os.makedirs(DATASET_DIR, exist_ok=True)
    with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
        json.dump(all_urls, f, ensure_ascii=False, indent=2)
    total_dur = time.time() - t0_total
    print(f"\n[{now()}] Đã cập nhật URLs vào {DATA_URLS_PATH}")
    print(f"Tổng thời gian crawl URL: {fmt_dur(total_dur)}")

✅ Tất cả categories đã có URL. Bỏ qua bước crawl URL.

  chinh-tri: 12506 URLs
  thoi-su: 12506 URLs
  kinh-doanh: 12506 URLs
  dan-toc-ton-giao: 3355 URLs
  the-thao: 12506 URLs
  giao-duc: 12506 URLs
  the-gioi: 12506 URLs
  doi-song: 12506 URLs
  van-hoa-giai-tri: 12506 URLs
  suc-khoe: 12506 URLs
  cong-nghe: 12506 URLs
  phap-luat: 12506 URLs
  oto-xe-may: 12506 URLs
  du-lich: 10683 URLs
  bat-dong-san: 12506 URLs
  ban-doc: 12506 URLs
  tuan-viet-nam: 11628 URLs
  bao-ve-nguoi-tieu-dung: 3873 URLs
  thi-truong-tieu-dung: 9332 URLs


---
## Section 3: Crawl Title và Content

Crawl tiêu đề và nội dung từng bài và lưu thành file `.parquet` trong `Dataset/`.

**Chỉ crawl những category chưa có file `.parquet`.**  
Nếu category đó chưa có URL trong `all_urls`, sẽ tự động crawl URL trước.

In [6]:
# Xac dinh category nao can crawl content
existing_parquet = (
    {f.replace(".parquet", "") for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")}
    if os.path.exists(DATASET_DIR) else set()
)
missing_content_cats = [cat for cat in CATEGORIES if cat not in existing_parquet]

if not missing_content_cats:
    print("✅ Tat ca categories da co du lieu parquet. Bo qua buoc crawl noi dung.\n")
    for pf in sorted(f for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")):
        print(f"  - {pf}")
else:
    t0_total = time.time()
    print(f"[{now()}] Can crawl content cho {len(missing_content_cats)} category:\n")
    for cat in missing_content_cats:
        has_url = bool(all_urls.get(cat))
        print(f"  • {cat}  [da co URL]" if has_url else f"  • {cat}  [chua co URL, se crawl URL truoc]")
    print()

    os.makedirs(DATASET_DIR, exist_ok=True)

    if "all_urls" not in dir():
        all_urls = json.load(open(DATA_URLS_PATH, encoding="utf-8")) if os.path.exists(DATA_URLS_PATH) else {}

    async def _crawl_category_async(category, urls):
        """Crawl content cho 1 category bang aiohttp + asyncio Semaphore."""
        out_path = os.path.join(DATASET_DIR, f"{category}.parquet")
        sem = asyncio.Semaphore(NUM_WORKERS)
        count = 0
        batch = []

        connector = aiohttp.TCPConnector(limit=NUM_WORKERS, ssl=False)
        timeout   = aiohttp.ClientTimeout(total=15)

        async def _one(url):
            async with sem:
                return url, await extract_content_async(session, url)

        async with aiohttp.ClientSession(
            headers=HEADERS, connector=connector, timeout=timeout
        ) as session:
            tasks = [asyncio.create_task(_one(url)) for url in urls]
            with pq.ParquetWriter(out_path, SCHEMA) as writer:
                with tqdm.tqdm(total=len(urls), desc=category) as pbar:
                    for coro in asyncio.as_completed(tasks):
                        _url, (title, content) = await coro
                        pbar.update(1)
                        if not title and not content:
                            continue
                        batch.append({"class": category, "url": _url, "title": title, "content": content})
                        count += 1
                        if len(batch) >= BATCH_SIZE:
                            flush_batch(writer, batch)
                flush_batch(writer, batch)

        return count, out_path

    async def _main():
        for i, category in enumerate(missing_content_cats, 1):
            urls = all_urls.get(category, [])
            prefix = f"[{now()}] [{i}/{len(missing_content_cats)}]"

            # Neu chua co URL, crawl URL truoc (dong bo)
            if not urls:
                print(f"\n{prefix} {category}: chua co URL → crawl URL truoc...")
                t0 = time.time()
                urls = get_urls_of_category(category, max_pages=MAX_PAGES)
                all_urls[category] = urls
                with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
                    json.dump(all_urls, f, ensure_ascii=False, indent=2)
                print(f"[{now()}] Da crawl duoc {len(urls)} URLs  ({fmt_dur(time.time() - t0)})")

            if not urls:
                print(f"[{now()}] Khong tim duoc URL nao cho {category}, bo qua")
                continue

            print(f"\n{prefix} Bat dau crawl content: {category}  ({len(urls)} URLs, {NUM_WORKERS} workers)")
            t0 = time.time()

            count, out_path = await _crawl_category_async(category, urls)

            dur = time.time() - t0
            if count == 0:
                os.remove(out_path)
                print(f"[{now()}] {category}: Khong co du lieu, bo qua")
            else:
                speed = len(urls) / dur
                print(f"[{now()}] ✓ {category}: {count} bai viet  ({fmt_dur(dur)}, ~{speed:.0f} URL/s)  → {out_path}")

        total_dur = time.time() - t0_total
        elapsed = str(datetime.timedelta(seconds=int(total_dur)))
        print(f"\n✅ Hoan tat crawl noi dung!  Tong thoi gian: {elapsed}")

    await _main()


[15:15:01] Can crawl content cho 18 category:

  • thoi-su  [da co URL]
  • kinh-doanh  [da co URL]
  • dan-toc-ton-giao  [da co URL]
  • the-thao  [da co URL]
  • giao-duc  [da co URL]
  • the-gioi  [da co URL]
  • doi-song  [da co URL]
  • van-hoa-giai-tri  [da co URL]
  • suc-khoe  [da co URL]
  • cong-nghe  [da co URL]
  • phap-luat  [da co URL]
  • oto-xe-may  [da co URL]
  • du-lich  [da co URL]
  • bat-dong-san  [da co URL]
  • ban-doc  [da co URL]
  • tuan-viet-nam  [da co URL]
  • bao-ve-nguoi-tieu-dung  [da co URL]
  • thi-truong-tieu-dung  [da co URL]


[15:15:01] [1/18] Bat dau crawl content: thoi-su  (12506 URLs, 32 workers)


thoi-su: 100%|██████████| 12506/12506 [03:46<00:00, 55.25it/s]


[15:18:47] ✓ thoi-su: 12506 bai viet  (3m 46s, ~55 URL/s)  → d:\VietNamNet-News-Classification\Dataset\thoi-su.parquet

[15:18:47] [2/18] Bat dau crawl content: kinh-doanh  (12506 URLs, 32 workers)


kinh-doanh: 100%|██████████| 12506/12506 [06:29<00:00, 32.14it/s]


[15:25:16] ✓ kinh-doanh: 12506 bai viet  (6m 29s, ~32 URL/s)  → d:\VietNamNet-News-Classification\Dataset\kinh-doanh.parquet

[15:25:16] [3/18] Bat dau crawl content: dan-toc-ton-giao  (3355 URLs, 32 workers)


...

[16:23:43] ✓ tuan-viet-nam: 11627 bai viet  (5m 18s, ~36 URL/s)  → d:\VietNamNet-News-Classification\Dataset\tuan-viet-nam.parquet

[16:23:43] [17/18] Bat dau crawl content: bao-ve-nguoi-tieu-dung  (3873 URLs, 32 workers)


bao-ve-nguoi-tieu-dung: 100%|██████████| 3873/3873 [01:43<00:00, 37.28it/s]


[16:25:27] ✓ bao-ve-nguoi-tieu-dung: 3873 bai viet  (1m 43s, ~37 URL/s)  → d:\VietNamNet-News-Classification\Dataset\bao-ve-nguoi-tieu-dung.parquet

[16:25:27] [18/18] Bat dau crawl content: thi-truong-tieu-dung  (9332 URLs, 32 workers)


thi-truong-tieu-dung: 100%|██████████| 9332/9332 [03:10<00:00, 48.93it/s]

[16:28:38] ✓ thi-truong-tieu-dung: 9332 bai viet  (3m 10s, ~49 URL/s)  → d:\VietNamNet-News-Classification\Dataset\thi-truong-tieu-dung.parquet

✅ Hoan tat crawl noi dung!  Tong thoi gian: 1:13:37


In [7]:
# ── Section 4: Kiểm tra chất lượng dữ liệu sau crawl ───────────────────────────
import os, json
import pyarrow.parquet as pq

if not os.path.exists(DATASET_DIR):
    print("❌ DATASET_DIR không tồn tại.")
else:
    rows = []
    empty_title_urls   = {}   # {category: [url, ...]}
    empty_content_urls = {}   # {category: [url, ...]}

    for cat in sorted(CATEGORIES):
        pf = os.path.join(DATASET_DIR, f"{cat}.parquet")
        if not os.path.exists(pf):
            rows.append((cat, 0, 0, 0, 0))
            continue

        schema   = pq.read_schema(pf)
        has_url  = "url" in schema.names
        cols     = ["url", "title", "content"] if has_url else ["title", "content"]
        tbl      = pq.read_table(pf, columns=cols)

        total    = len(tbl)
        titles   = tbl["title"].to_pylist()
        contents = tbl["content"].to_pylist()
        urls_col = tbl["url"].to_pylist() if has_url else [None] * total

        no_title = no_cont = no_both = 0
        et_urls  = []
        ec_urls  = []
        for u, t, c in zip(urls_col, titles, contents):
            miss_t = not str(t).strip()
            miss_c = not str(c).strip()
            if miss_t:
                no_title += 1
                if u: et_urls.append(u)
            if miss_c:
                no_cont += 1
                if u: ec_urls.append(u)
            if miss_t and miss_c:
                no_both += 1

        if et_urls: empty_title_urls[cat]   = et_urls
        if ec_urls: empty_content_urls[cat] = ec_urls
        rows.append((cat, total, no_title, no_cont, no_both))

    # ── In bảng thống kê
    print()
    H = f"  {'Category':<28}  {'Tổng':>7}  {'Thiếu title':>12}  {'Thiếu content':>14}  {'Thiếu cả 2':>10}"
    print(H)
    print("-" * len(H))
    for cat, total, no_t, no_c, no_both in rows:
        if total == 0:
            print(f"  {cat:<28}  (chưa crawl)")
            continue
        flag = "  ⚠️" if (no_t > 0 or no_c > 0) else ""
        print(f"  {cat:<28}  {total:>7,}  {no_t:>8,} ({no_t/total*100:4.1f}%)"
              f"  {no_c:>10,} ({no_c/total*100:4.1f}%)  {no_both:>6,}{flag}")
    print("-" * len(H))

    tot_total = sum(r[1] for r in rows)
    tot_no_t  = sum(r[2] for r in rows)
    tot_no_c  = sum(r[3] for r in rows)
    tot_no_b  = sum(r[4] for r in rows)
    if tot_total > 0:
        print(f"  {'TỔNG CỘNG':<28}  {tot_total:>7,}"
              f"  {tot_no_t:>8,} ({tot_no_t/tot_total*100:4.1f}%)"
              f"  {tot_no_c:>10,} ({tot_no_c/tot_total*100:4.1f}%)  {tot_no_b:>6,}")

    # ── Export debug JSON
    et_path = os.path.join(DATASET_DIR, "data_URLs_empty_title.json")
    ec_path = os.path.join(DATASET_DIR, "data_URLs_empty_content.json")
    with open(et_path, "w", encoding="utf-8") as f:
        json.dump(empty_title_urls, f, ensure_ascii=False, indent=2)
    with open(ec_path, "w", encoding="utf-8") as f:
        json.dump(empty_content_urls, f, ensure_ascii=False, indent=2)

    total_et = sum(len(v) for v in empty_title_urls.values())
    total_ec = sum(len(v) for v in empty_content_urls.values())
    print()
    print(f"  ℹ️  Thiếu title  : {tot_no_t:,} bài  →  {total_et:,} URL đã lưu → data_URLs_empty_title.json")
    print(f"  ℹ️  Thiếu content: {tot_no_c:,} bài  →  {total_ec:,} URL đã lưu → data_URLs_empty_content.json")
    print(f"  ℹ️  Thiếu cả 2  : {tot_no_b:,} bài  (sẽ bị loại khi train)")
    if tot_no_t > total_et or tot_no_c > total_ec:
        print("  ⚠️   File parquet cũ không có cột url — re-crawl để có đầy đủ URL debug.")



  Category                         Tổng   Thiếu title   Thiếu content  Thiếu cả 2
---------------------------------------------------------------------------------
  ban-doc                        12,506         0 ( 0.0%)           1 ( 0.0%)       0  ⚠️
  bao-ve-nguoi-tieu-dung          3,873         0 ( 0.0%)           0 ( 0.0%)       0
  bat-dong-san                   12,506         0 ( 0.0%)           0 ( 0.0%)       0
  chinh-tri                      12,506         0 ( 0.0%)           5 ( 0.0%)       0  ⚠️
  cong-nghe                      12,506         0 ( 0.0%)           0 ( 0.0%)       0
  dan-toc-ton-giao                3,355         0 ( 0.0%)           0 ( 0.0%)       0
  doi-song                       12,506         0 ( 0.0%)           0 ( 0.0%)       0
  du-lich                        10,683         0 ( 0.0%)           0 ( 0.0%)       0
  giao-duc                       12,506         0 ( 0.0%)           0 ( 0.0%)       0
  kinh-doanh                     12,506         0 ( 0